# Day 1: Pokémon TCG simulator

Team A | Rishitha Gogineni | 21 November 2026

Run the simulator from the Kaggle sample package, inspect its observations, and check that random agents can finish a battle. Save observation examples for feature engineering and record the runtime used.

This notebook contains the completed Colab run. It uses individual battles with the sample deck. Python seeds control action sampling; the simulator's own randomness has not been verified as seeded.


## 1. Environment

Record Python, platform details, installed package versions, and the output folder in `environment.json`.


In [1]:
import hashlib
import importlib.metadata
import json
import os
import platform
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
output_dir = Path('/content/ptcg_day1_results') / RUN_ID
output_dir.mkdir(parents=True, exist_ok=False)
environment_path = output_dir / 'environment.json'
package_versions = {}
for distribution in ('kaggle-environments', 'numpy', 'pandas', 'torch'):
    try:
        package_versions[distribution] = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        package_versions[distribution] = None

environment = {
    'run_id': RUN_ID,
    'started_at_utc': utc_now(),
    'python_version': sys.version,
    'system': platform.system(),
    'platform': platform.platform(),
    'machine': platform.machine(),
    'logical_cpu_count': os.cpu_count(),
    'installed_package_versions': package_versions,
    'competition_url': 'https://www.kaggle.com/competitions/the-pokemon-company-ptcg-ai-battle-challenge-playground',
    'archive': None,
    'sdk_files': [],
    'sdk_imported': False,
    'simulator_seed_verified': False,
    'unit': 'individual_battle',
    'turn_band_definition': {'setup': 'turn <= 0', 'early': 'turns 1-4', 'middle': 'turns 5-12', 'later': 'turn >= 13'},
    'sampling_note': 'Turn bands are sampling labels, not inferred strategic game phases.',
}
write_json(environment_path, environment)
print('Python:', sys.version.split()[0])
print('System:', platform.system(), '| Machine:', platform.machine())
print('Results folder:', output_dir)
assert platform.system() == 'Linux', 'Use a Linux Colab runtime.'
assert platform.machine() in ('x86_64', 'amd64', 'aarch64', 'arm64'), 'Unsupported architecture.'


Python: 3.13.18
System: Linux | Machine: x86_64
Results folder: /content/ptcg_day1_results/20261121T214059Z_351ac356


## 2. Competition files

Upload `the-pokemon-company-ptcg-ai-battle-challenge-playground.zip`. Record its filename, size, and SHA 256 hash.


In [2]:
from google.colab import files

expected_name = 'the-pokemon-company-ptcg-ai-battle-challenge-playground.zip'
archive_path = Path('/content') / expected_name
if not archive_path.exists():
    archives = sorted(Path('/content').glob('the-pokemon-company-ptcg-ai-battle-challenge-playground*.zip'))
    if len(archives) == 1:
        archive_path = archives[0]
    elif len(archives) > 1:
        raise ValueError('Multiple competition ZIPs found. Keep one intended version or rename it to ' + expected_name)
    else:
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if len(zip_names) != 1:
            raise ValueError('Upload exactly one official competition ZIP.')
        archive_path = Path('/content') / zip_names[0]

environment['archive'] = {
    'filename': archive_path.name,
    'bytes': archive_path.stat().st_size,
    'sha256': sha256_file(archive_path),
}
write_json(environment_path, environment)
print('Archive:', archive_path.name)
print('SHA-256:', environment['archive']['sha256'])


Saving the-pokemon-company-ptcg-ai-battle-challenge-playground.zip to the-pokemon-company-ptcg-ai-battle-challenge-playground.zip
Archive: the-pokemon-company-ptcg-ai-battle-challenge-playground.zip
SHA-256: 6fe557e77ca21b86dea9056d7f4531cbeb328abc1b25aecf9c3a4e2b7d5b2c87


## 3. Sample package

Extract the sample agent, deck, and simulator files. Keep each run in a separate folder and record the SDK file hashes.


In [3]:
import zipfile

sample_dir = Path('/content/ptcg_sample') / RUN_ID
sample_dir.mkdir(parents=True, exist_ok=False)
prefix = 'sample_submission/sample_submission/sample_submission/'
with zipfile.ZipFile(archive_path) as archive:
    members = [name for name in archive.namelist() if name.startswith(prefix) and not name.endswith('/')]
    if not members:
        raise ValueError('Expected sample package not found. Inspect this ZIP version before changing the prefix.')
    for name in members:
        relative = Path(name[len(prefix):])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Unexpected path in ZIP: ' + name)
        destination = sample_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(name))

for required in ('main.py', 'deck.csv', 'cg/game.py'):
    assert (sample_dir / required).is_file(), required
environment['sdk_identity'] = 'bundled sample SDK; file hashes identify the exact version'
environment['sdk_files'] = [
    {'path': str(path.relative_to(sample_dir)), 'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
    for path in sorted((sample_dir / 'cg').rglob('*')) if path.is_file()
]
environment['sample_agent_sha256'] = sha256_file(sample_dir / 'main.py')
environment['deck_file_sha256'] = sha256_file(sample_dir / 'deck.csv')
write_json(environment_path, environment)
print('Sample files:', sorted(path.name for path in sample_dir.iterdir()))
print('Recorded SDK files:', len(environment['sdk_files']))


Sample files: ['cg', 'deck.csv', 'main.py']
Recorded SDK files: 9


## 4. Load the simulator

Import the battle functions and check that the sample deck contains 60 cards. Use a fresh Colab session when rerunning the notebook.


In [4]:
import traceback

# Avoid accidentally reusing an engine imported from another extraction/run.
if 'cg' in sys.modules:
    raise RuntimeError('The cg package is already loaded. Restart the Colab session and run all cells again.')
sys.path.insert(0, str(sample_dir))
try:
    import cg.sim as cg_sim
    from cg.game import battle_start, battle_select, battle_finish
    deck = [int(line.strip()) for line in (sample_dir / 'deck.csv').read_text().splitlines() if line.strip()]
    assert len(deck) == 60, 'Expected 60 deck entries.'
    environment['sdk_imported'] = True
    environment['native_library'] = Path(cg_sim.lib_path).name
    environment['deck_entries'] = len(deck)
    print('Sample deck size:', len(deck))
    print('Distinct card IDs:', len(set(deck)))
except Exception as error:
    environment['import_error'] = {'type': type(error).__name__, 'message': str(error), 'traceback': traceback.format_exc()}
    raise
finally:
    write_json(environment_path, environment)


Sample deck size: 60
Distinct card IDs: 9


## 5. Inspect an observation

Check the current player, turn, selection type, and available options. Save the inspection result and release the battle when finished.


In [5]:
inspection = {'status': 'not_run', 'observed_at_utc': utc_now()}
try:
    observation, start_data = battle_start(deck, deck)
    if observation is None:
        raise RuntimeError(f'Battle did not start: player={start_data.errorPlayer}, type={start_data.errorType}')
    current = observation.get('current') or {}
    selection = observation.get('select') or {}
    inspection.update({
        'status': 'passed',
        'observation_keys': list(observation),
        'turn': current.get('turn'),
        'acting_player': current.get('yourIndex'),
        'selection_type': selection.get('type'),
        'selection_context': selection.get('context'),
        'min_choices': selection.get('minCount'),
        'max_choices': selection.get('maxCount'),
        'option_count': len(selection.get('option', [])),
    })
    print(json.dumps(inspection, indent=2))
except Exception as error:
    inspection.update(status='error', error_type=type(error).__name__, error_message=str(error), traceback=traceback.format_exc())
    print('Inspection failed; saved the error. The test runner below records its own attempts.')
finally:
    try:
        if cg_sim.Battle.battle_ptr:
            battle_finish()
    except Exception as error:
        inspection['status'] = 'cleanup_error'
        inspection['cleanup_error'] = {'type': type(error).__name__, 'message': str(error), 'traceback': traceback.format_exc()}
    finally:
        cg_sim.Battle.battle_ptr = None
        write_json(output_dir / 'inspection.json', inspection)


{
  "status": "passed",
  "observed_at_utc": "2026-11-21T21:41:31.341553+00:00",
  "observation_keys": [
    "select",
    "logs",
    "current",
    "search_begin_input"
  ],
  "turn": 0,
  "acting_player": 0,
  "selection_type": 9,
  "selection_context": 41,
  "min_choices": 1,
  "max_choices": 1,
  "option_count": 2
}


## 6. Random battle and observation sampling

Run one random against random battle. Save up to 16 examples in each turn band: setup (turn 0), early (1 to 4), middle (5 to 12), and later (13+). Within a band, keep at most two examples per selection type/context combination. These bands organize sampling; they are not strategic game phases.

Each example includes the acting player's observation, chosen option indices, and whether the engine accepted the choice. The opaque `search_begin_input` field is excluded. Samples are for debugging and feature design, not expert demonstrations or complete trajectories.

Save checkpoints every 100 selections and write partial results if a Python error occurs. A native engine crash may leave only the last checkpoint.


In [6]:
import copy
import csv
import random
import time
from collections import Counter

TURN_BANDS = ('setup', 'early', 'middle', 'later')
SMOKE_TEST_SEEDS = (10, 20, 30)
MAX_STEPS = 10000
SAMPLES_PER_BAND = 16
CHECKPOINT_EVERY_STEPS = 100
environment['test_configuration'] = {
    'initial_python_seed': 42, 'smoke_python_seeds': list(SMOKE_TEST_SEEDS),
    'max_steps': MAX_STEPS, 'samples_per_turn_band': SAMPLES_PER_BAND,
    'max_examples_per_band_type_context': 2,
    'checkpoint_every_steps': CHECKPOINT_EVERY_STEPS,
}
write_json(environment_path, environment)

def turn_band(turn):
    if turn <= 0:
        return 'setup'
    if turn <= 4:
        return 'early'
    if turn <= 12:
        return 'middle'
    return 'later'

def coverage_key(band, selection):
    return json.dumps([band, selection.get('type'), selection.get('context')])

def sorted_samples(buckets):
    return sorted((record for bucket in buckets.values() for record in bucket), key=lambda record: record['step'])

def retain_sample(buckets, record, rng, per_band):
    if per_band <= 0:
        return
    bucket = buckets[record['turn_band']]
    key = record['coverage_key']
    same = [index for index, sample in enumerate(bucket) if sample['coverage_key'] == key]
    if len(same) >= 2:
        # Keep one early example, optionally refresh the second with a later one.
        if rng.random() < 0.25:
            bucket[same[-1]] = record
        return
    if len(bucket) < per_band:
        bucket.append(record)
        return
    if same:
        return
    # A new context replaces a duplicate context first; otherwise replace one slot.
    counts = Counter(sample['coverage_key'] for sample in bucket)
    duplicates = [index for index, sample in enumerate(bucket) if counts[sample['coverage_key']] > 1]
    bucket[rng.choice(duplicates or list(range(len(bucket))))] = record

def run_random_battle(run_name, seed=42, max_steps=MAX_STEPS, samples_per_band=SAMPLES_PER_BAND):
    run_dir = output_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=False)
    action_rng = random.Random(seed)
    sample_rng = random.Random(seed + 1000003)
    buckets = {band: [] for band in TURN_BANDS}
    observed = Counter()
    started = time.perf_counter()
    summary = {
        'run_name': run_name, 'python_seed': seed, 'status': 'running',
        'finished': False, 'result_code': None, 'steps': 0, 'seconds': 0.0,
        'started_at_utc': utc_now(), 'error_type': '', 'error_message': '',
        'traceback_file': '', 'cleanup_error': '', 'last_attempt': None,
    }

    def checkpoint():
        summary['seconds'] = round(time.perf_counter() - started, 3)
        saved = sorted_samples(buckets)
        write_json(run_dir / 'result.json', summary)
        write_json(run_dir / 'sample_observations.json', saved)
        write_json(run_dir / 'coverage.json', {
            'observed_by_band_type_context': dict(observed),
            'saved_by_band_type_context': dict(Counter(record['coverage_key'] for record in saved)),
            'saved_by_turn_band': {band: len(buckets[band]) for band in TURN_BANDS},
            'saved_total': len(saved), 'max_samples_per_band': samples_per_band,
            'turn_band_definitions': environment['turn_band_definition'],
            'note': 'Observed combinations are not a claim that all possible decision types were exercised.',
        })

    checkpoint()
    try:
        obs, start = battle_start(deck, deck)
        if obs is None:
            raise RuntimeError(f'Battle start error: player={start.errorPlayer}, type={start.errorType}')
        for step in range(max_steps + 1):
            state = obs.get('current')
            if state is None:
                raise RuntimeError('Missing current state in the direct battle API.')
            result_code = state.get('result', -1)
            if result_code != -1:
                summary.update(status='passed', finished=True, result_code=result_code)
                break
            if step == max_steps:
                summary.update(status='step_limit', error_type='StepLimit', error_message=f'No terminal state after {max_steps} selections.')
                break
            selection = obs.get('select')
            if selection is None:
                raise RuntimeError(f'Missing legal choices at step {step}.')
            options = selection.get('option', [])
            minimum = selection['minCount']
            maximum = selection['maxCount']
            if not (0 <= minimum <= maximum <= len(options)):
                raise ValueError(f'Invalid selection counts at step {step}.')
            indices = action_rng.sample(range(len(options)), maximum)
            band = turn_band(state.get('turn', 0))
            key = coverage_key(band, selection)
            observed[key] += 1
            record = {
                'run_name': run_name, 'step': step, 'turn': state.get('turn'),
                'turn_band': band, 'acting_player': state.get('yourIndex'),
                'selection_type': selection.get('type'), 'selection_context': selection.get('context'),
                'min_choices': minimum, 'max_choices': maximum, 'option_count': len(options),
                'selected_option_indices': indices, 'selection_accepted': False, 'coverage_key': key,
                'observation': {field: copy.deepcopy(obs[field]) for field in ('logs', 'current', 'select') if field in obs},
            }
            # Keep compact last-attempt metadata even when this observation is not sampled.
            summary['last_attempt'] = {field: value for field, value in record.items() if field != 'observation'}
            retain_sample(buckets, record, sample_rng, samples_per_band)
            obs = battle_select(indices)
            record['selection_accepted'] = True
            summary['last_attempt']['selection_accepted'] = True
            summary['steps'] = step + 1
            if (step + 1) % CHECKPOINT_EVERY_STEPS == 0:
                checkpoint()
    except KeyboardInterrupt:
        summary.update(status='interrupted', error_type='KeyboardInterrupt', error_message='User interrupted this battle.')
        raise
    except Exception as error:
        summary.update(status='error', error_type=type(error).__name__, error_message=str(error), traceback_file=f'{run_name}/error_traceback.txt')
        (run_dir / 'error_traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
    finally:
        try:
            if cg_sim.Battle.battle_ptr:
                battle_finish()
        except Exception as error:
            summary['cleanup_error'] = f'{type(error).__name__}: {error}'
            if summary['status'] == 'passed':
                summary['status'] = 'cleanup_error'
            (run_dir / 'cleanup_traceback.txt').write_text(traceback.format_exc(), encoding='utf-8')
        finally:
            cg_sim.Battle.battle_ptr = None
            if summary['status'] != 'running':
                summary['finished_at_utc'] = utc_now()
            checkpoint()
    return summary, sorted_samples(buckets)

result, samples = run_random_battle('initial_battle', seed=42)
print({key: result[key] for key in ('run_name', 'status', 'finished', 'result_code', 'steps', 'seconds', 'error_type', 'error_message')})
print('Collected observations:', len(samples))
if result['status'] != 'passed':
    print('This battle did not pass. Inspect its saved result/error files; later smoke tests can still run.')


{'run_name': 'initial_battle', 'status': 'passed', 'finished': True, 'result_code': 0, 'steps': 34, 'seconds': 0.017, 'error_type': '', 'error_message': ''}
Collected observations: 17


## 7. Observation handoff

Save the initial battle's samples to `sample_observations.json`. Read each record's `observation` field for the SDK data; the other fields describe the decision and selected options. Use `current.yourIndex` for the acting player's perspective.

The initial battle saved 17 examples. The smoke test folders below contain additional samples, including later turns.


In [7]:
observation_path = output_dir / 'sample_observations.json'
write_json(observation_path, samples)
print('Saved:', observation_path)
print('Saved turn bands:', dict(Counter(sample['turn_band'] for sample in samples)))
print('Saved selection types:', dict(Counter(str(sample['selection_type']) for sample in samples)))
print('Record fields:', list(samples[0]) if samples else [])
print("Inspect each run's coverage.json for observed versus saved combinations.")


Saved: /content/ptcg_day1_results/20261121T214059Z_351ac356/sample_observations.json
Saved turn bands: {'setup': 3, 'early': 5, 'middle': 9}
Saved selection types: {'9': 1, '1': 10, '0': 4, '4': 2}
Record fields: ['run_name', 'step', 'turn', 'turn_band', 'acting_player', 'selection_type', 'selection_context', 'min_choices', 'max_choices', 'option_count', 'selected_option_indices', 'selection_accepted', 'coverage_key', 'observation']
Inspect each run's coverage.json for observed versus saved combinations.


## 8. Smoke tests

Run three more battles using Python seeds 10, 20, and 30. Save each result before continuing, along with any error details. These tests check execution, not playing strength.


In [8]:
CSV_FIELDS = ['run_name', 'python_seed', 'status', 'finished', 'result_code', 'steps', 'seconds', 'error_type', 'error_message', 'traceback_file', 'cleanup_error']
results_path = output_dir / 'smoke_test_results.csv'
all_results_path = output_dir / 'all_battle_results.json'
results = []

def save_aggregate():
    temporary = results_path.with_suffix('.csv.tmp')
    with temporary.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    temporary.replace(results_path)
    write_json(all_results_path, [result] + results)

save_aggregate()
for test_seed in SMOKE_TEST_SEEDS:
    test_result, _ = run_random_battle(f'smoke_seed_{test_seed}', seed=test_seed)
    results.append(test_result)
    save_aggregate()
    print({field: test_result[field] for field in ('run_name', 'status', 'finished', 'steps', 'seconds', 'error_type', 'error_message')})

passed = sum(row['status'] == 'passed' and row['finished'] for row in results)
day1_status = {
    'sdk_imported': environment['sdk_imported'],
    'inspection_status': inspection['status'],
    'initial_battle_passed': result['status'] == 'passed' and result['finished'],
    'smoke_tests_expected': len(SMOKE_TEST_SEEDS),
    'smoke_tests_recorded': len(results),
    'smoke_tests_passed': passed,
    'runtime_checks_passed': (
        environment['sdk_imported'] and inspection['status'] == 'passed'
        and result['status'] == 'passed' and result['finished']
        and passed == len(SMOKE_TEST_SEEDS)
    ),
    'representative_sample_count': len(samples),
    'handoff_to_kavya_confirmed': False,
    'note': 'Runtime checks do not establish BO3 performance, Kaggle validation or complete decision-type coverage.',
}
write_json(output_dir / 'day1_status.json', day1_status)
environment['last_updated_at_utc'] = utc_now()
environment['initial_battle_seconds'] = result['seconds']
environment['smoke_test_seconds'] = [row['seconds'] for row in results]
environment['total_recorded_battle_seconds'] = round(result['seconds'] + sum(row['seconds'] for row in results), 3)
write_json(environment_path, environment)
print(f'Smoke tests passed: {passed}/{len(SMOKE_TEST_SEEDS)}')
print('Overall runtime checks:', 'PASSED' if day1_status['runtime_checks_passed'] else 'INCOMPLETE / FAILED - inspect saved errors')
print('Saved:', results_path)


{'run_name': 'smoke_seed_10', 'status': 'passed', 'finished': True, 'steps': 31, 'seconds': 0.017, 'error_type': '', 'error_message': ''}
{'run_name': 'smoke_seed_20', 'status': 'passed', 'finished': True, 'steps': 119, 'seconds': 0.069, 'error_type': '', 'error_message': ''}
{'run_name': 'smoke_seed_30', 'status': 'passed', 'finished': True, 'steps': 74, 'seconds': 0.042, 'error_type': '', 'error_message': ''}
Smoke tests passed: 3/3
Overall runtime checks: PASSED
Saved: /content/ptcg_day1_results/20261121T214059Z_351ac356/smoke_test_results.csv


## 9. Save the results

Download the run folder as a ZIP containing environment details, observations, coverage, and test results. The competition files and simulator binaries are not included.

Also download the executed `.ipynb` from Colab to retain the cell outputs.


In [9]:
import zipfile
from google.colab import files

bundle_path = output_dir.parent / f'{RUN_ID}_results.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output_dir.rglob('*')):
        if path.is_file() and path.suffix != '.tmp':
            bundle.write(path, arcname=str(Path(RUN_ID) / path.relative_to(output_dir)))
print('Download:', bundle_path.name)
print('Includes saved successes, failures and available partial observations.')
files.download(str(bundle_path))


Download: 20261121T214059Z_351ac356_results.zip
Includes saved successes, failures and available partial observations.


## Results and next steps

The simulator imported successfully. The inspection, initial battle, and all three smoke tests passed without recorded errors.

Initial battle: 34 selections, 0.017 seconds and 17 saved examples.

Seed 10: 31 selections, 0.017 seconds and 14 saved examples.

Seed 20: 119 selections, 0.069 seconds and 26 saved examples.

Seed 30: 74 selections, 0.042 seconds and 22 saved examples.


The four battles produced 79 distinct saved examples, covering setup, early, middle, and later turns. The top level sample file contains only the initial battle's 17 examples; the remaining samples are in their respective battle folders. Timings include sampling and file writes and are not a throughput benchmark.

Next, share the observation handoff with Kavya, implement the reusable runner, and build the baseline agent. Receipt of the handoff is still pending. These results do not establish best of three performance, complete decision coverage, Kaggle submission compatibility, or a leaderboard score.

References: [competition](https://www.kaggle.com/competitions/the-pokemon-company-ptcg-ai-battle-challenge-playground) and [simulator documentation](https://matsuoinstitute.github.io/cabt/).
